In [ ]:
#installing libraries
%pip install rasterio matplotlib geopandas
%pip install scikit-learn pandas numpy shapely

In [ ]:
#import libraries
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
import matplotlib.pyplot as plt
from shapely.geometry import box
import geopandas as gpd
import numpy as np
import geopandas as gpd
import os

Changing projection of the Land Use dataset

In [ ]:
raster_path = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\2022 Land Use\land use 22.tif"
output_path = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\2022 Land Use\LandUseWGS84.tif"

# define target projection
dst_crs = "EPSG:4326"

# Mainland US bounding box in Web Mercator (meters)
mainland_us_bounds = (-13892000, 2870000, -7450000, 6330000)

with rasterio.open(raster_path) as src:
    print("Original CRS:", src.crs)

    # Clip window from bounding box
    window = from_bounds(*mainland_us_bounds, transform=src.transform)
    src_data = src.read(1, window=window)
    src_transform = src.window_transform(window)

    # Calculate destination transform, width, and height
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src_data.shape[1], src_data.shape[0], *mainland_us_bounds
    )

    # Prepare destination array and metadata
    dst_data = np.empty((height, width), dtype=src_data.dtype)
    dst_meta = src.meta.copy()
    dst_meta.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height,
        "driver": "GTiff"
    })

    # Reproject
    reproject(
        source=src_data,
        destination=dst_data,
        src_transform=src_transform,
        src_crs=src.crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )

# save as new GeoTIFF
with rasterio.open(output_path, "w", **dst_meta) as dst:
    dst.write(dst_data, 1)

print(f"Reprojected raster saved to:\n{output_path}")


test landuse visualization 

In [ ]:
# open the reprojected raster
with rasterio.open(output_path) as dataset:
    data = dataset.read(1)  # read the first band
    plt.figure(figsize=(10, 8))
    plt.imshow(data, cmap='terrain')
    plt.colorbar(label='Land Use Value', shrink=0.5)
    plt.title("Reprojected Land Use Raster (EPSG:4326)")
    plt.axis('off')
    plt.show()

# Import Datasets

In [ ]:
# paths to datasets
land_use = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\2022 Land Use\LandUseWGS84.tif"
substations = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\Electric_Substations\Electric_Substations.shp"
hp_solarpanels = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\HPspSHP\uspvdb_v2_0_20240801.shp"
ghi_annual =  r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_ghi\nsrdbv3_ghi\Annual GHI\nsrdb3_ghi.tif"
dni_annual = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Annual DNI\nsrdb3_dni.tif"
power_plants = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\PowerPlants\PowerPlants_US_EIA.shp"
us_shapefile = r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\tl_2024_us_state\tl_2024_us_state.shp"

### Prepare the spatial grid

In [ ]:
# load US shapefile and get bounds
us = gpd.read_file(us_shapefile)
minx, miny, maxx, maxy = us.total_bounds

# create a square grid in original CRS
grid_size = 0.1  # ~10km
grid_cells = []
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        grid_cells.append(box(x, y, x + grid_size, y + grid_size))
        y += grid_size
    x += grid_size

grid = gpd.GeoDataFrame(geometry=grid_cells, crs=us.crs)
grid = gpd.overlay(grid, us, how='intersection')  # Clip to US boundary

### Extract Raster Features: GHI, DNI, Land Use

In [ ]:
# 3. Sample raster features (after reprojecting grid to match raster CRS)
import rasterio
from rasterio.features import geometry_mask

def sample_raster(raster_path, geo_df):
    with rasterio.open(raster_path) as src:
        values = []
        for geom in geo_df.geometry:
            try:
                out_image, out_transform = rasterio.mask.mask(src, [geom.buffer(0)], crop=True)
                masked_data = out_image[0][out_image[0] != src.nodata]
                values.append(masked_data.mean() if masked_data.size > 0 else np.nan)
            except:
                values.append(np.nan)
        return values

# Detect raster CRS (EPSG:4326)
with rasterio.open(ghi_annual) as src:
    raster_crs = src.crs

# Reproject grid temporarily to raster CRS
grid_for_sampling = grid.to_crs(raster_crs)

# Sample rasters using reprojected grid
ghi_vals = sample_raster(ghi_annual, grid_for_sampling)
dni_vals = sample_raster(dni_annual, grid_for_sampling)
landuse_vals = sample_raster(land_use, grid_for_sampling)

# Assign sampled values back to original grid using .loc for index alignment
grid.loc[:, 'GHI'] = ghi_vals
grid.loc[:, 'DNI'] = dni_vals
grid.loc[:, 'LandUse'] = landuse_vals

# 4. Load vector data
subs = gpd.read_file(substations)
solars = gpd.read_file(hp_solarpanels)

# 5. Reproject everything to a **projected CRS** for distance calculations
projected_crs = "EPSG:5070"
grid = grid.to_crs(projected_crs)
subs = subs.to_crs(projected_crs)
solars = solars.to_crs(projected_crs)


### Extract Vector Features: Distance to Substations & Solar Plants

In [ ]:
# 6. Calculate distances from centroids to nearest substations/solar panels
from shapely.ops import nearest_points
from shapely.geometry import Point

grid['centroid'] = grid.centroid

def nearest_distance(centroids, points):
    return centroids.apply(lambda x: points.distance(x).min())

grid['dist_to_substation'] = nearest_distance(grid['centroid'], subs.geometry)
grid['dist_to_solar'] = nearest_distance(grid['centroid'], solars.geometry)

### Run Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Drop rows with missing values
features = grid[['GHI', 'DNI', 'LandUse', 'dist_to_substation', 'dist_to_solar']].dropna()
scaled_features = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=5, random_state=42)
grid.loc[features.index, 'cluster'] = kmeans.fit_predict(scaled_features)

### Visualize Clusters

In [ ]:
import matplotlib.pyplot as plt

grid.plot(column='cluster', cmap='tab10', legend=True, figsize=(12, 8))
plt.title("Potential Solar Site Clusters")
plt.show()